In [1]:
import os
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-11.0.31.11-hotspot"
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] += r";C:\hadoop\bin"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("RetailPipeline") \
    .config("spark.jars.packages", 
            "io.delta:delta-core_2.12:2.4.0") \
    .config("spark.sql.extensions", 
            "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", 
            "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

print("SparkSession ready ✅")

SparkSession ready ✅


In [14]:
import os
import sys

os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-11.0.31.11-hotspot"
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] += r";C:\hadoop\bin"

# Tell Spark to use the same Python that's running this notebook
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print(f"Using Python: {sys.executable}")

Using Python: c:\Users\Admin\anaconda3\python.exe


In [2]:

SILVER_PATH = r"D:\DS\visualization\retail_pipeline\data\silver\data_8f810161-25f8-4fea-bdff-ef679dc02b9a_ee3b5a04-bae9-4656-9aaa-9c90e8fedc9a.parquet"
GOLD_PATH   = r"D:\DS\visualization\retail_pipeline\data\gold"

df = spark.read.parquet(SILVER_PATH)

print(f"Row count: {df.count()}")
print(f"Columns: {df.columns}")
df.printSchema()
df.show(5)

Row count: 525461
Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer_ID', 'Country']
root
 |-- Invoice: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- Price: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Country: string (nullable = true)

+-------+---------+--------------------+--------+----------------+-----+-----------+--------------+
|Invoice|StockCode|         Description|Quantity|     InvoiceDate|Price|Customer_ID|       Country|
+-------+---------+--------------------+--------+----------------+-----+-----------+--------------+
| 489434|    85048|15CM CHRISTMAS GL...|      12|01-12-2009 07:45| 6.95|      13085|United Kingdom|
| 489434|   79323P|  PINK CHERRY LIGHTS|      12|01-12-2009 07:45| 6.75|      13085|United Kingdom|
| 489434|   79323W| WHITE CHERRY LIGHTS|

In [3]:
# 1. Drop nulls in critical columns
df_clean = df.dropna(subset=["Invoice", "Customer_ID", "StockCode"])

# 2. Remove cancelled orders (Invoice starts with 'C')
df_clean = df_clean.filter(~F.col("Invoice").startswith("C"))

# 3. Remove negative Quantity or Price
df_clean = df_clean.filter(
    (F.col("Quantity") > 0) & (F.col("Price") > 0)
)

# 4. Add TotalAmount column
df_clean = df_clean.withColumn(
    "TotalAmount", F.col("Quantity") * F.col("Price")
)

# 5. Cast InvoiceDate to timestamp
df_clean = df_clean.withColumn(
    "InvoiceDate", F.to_timestamp("InvoiceDate")
)

# 6. Add Year and Month columns
df_clean = df_clean \
    .withColumn("Year", F.year("InvoiceDate")) \
    .withColumn("Month", F.month("InvoiceDate"))

print(f"Clean row count: {df_clean.count()}")
df_clean.show(5)

Clean row count: 322476
+-------+---------+--------------------+--------+-----------+-----+-----------+--------------+------------------+----+-----+
|Invoice|StockCode|         Description|Quantity|InvoiceDate|Price|Customer_ID|       Country|       TotalAmount|Year|Month|
+-------+---------+--------------------+--------+-----------+-----+-----------+--------------+------------------+----+-----+
| 489434|    85048|15CM CHRISTMAS GL...|      12|       null| 6.95|      13085|United Kingdom|              83.4|null| null|
| 489434|   79323P|  PINK CHERRY LIGHTS|      12|       null| 6.75|      13085|United Kingdom|              81.0|null| null|
| 489434|   79323W| WHITE CHERRY LIGHTS|      12|       null| 6.75|      13085|United Kingdom|              81.0|null| null|
| 489434|    22041|RECORD FRAME 7" S...|      48|       null|  2.1|      13085|United Kingdom|100.80000000000001|null| null|
| 489434|    21232|STRAWBERRY CERAMI...|      24|       null| 1.25|      13085|United Kingdom|       

In [4]:
gold_monthly_revenue = df_clean \
    .groupBy("Year", "Month", "Country") \
    .agg(
        F.round(F.sum("TotalAmount"), 2).alias("TotalRevenue"),
        F.countDistinct("Invoice").alias("TotalOrders"),
        F.countDistinct("Customer_ID").alias("UniqueCustomers")
    ).orderBy("Year", "Month", F.desc("TotalRevenue"))

gold_monthly_revenue.show(10)

+----+-----+---------------+------------+-----------+---------------+
|Year|Month|        Country|TotalRevenue|TotalOrders|UniqueCustomers|
+----+-----+---------------+------------+-----------+---------------+
|null| null| United Kingdom|  6453276.06|      17245|           3926|
|null| null|           EIRE|   320431.56|        314|              5|
|null| null|    Netherlands|   230096.71|        135|             22|
|null| null|        Germany|   176704.79|        344|             67|
|null| null|         France|   114805.39|        233|             47|
|null| null|          Spain|    44483.73|         65|             25|
|null| null|    Switzerland|    39362.59|         40|             14|
|null| null|         Sweden|    38943.68|         68|             16|
|null| null|      Australia|     27007.0|         40|             15|
|null| null|Channel Islands|    22341.64|         30|             11|
+----+-----+---------------+------------+-----------+---------------+
only showing top 10 

In [5]:
gold_top_products = df_clean \
    .groupBy("StockCode", "Description") \
    .agg(
        F.round(F.sum("TotalAmount"), 2).alias("TotalRevenue"),
        F.sum("Quantity").alias("TotalQuantitySold")
    ).orderBy(F.desc("TotalRevenue"))

gold_top_products.show(10)

+---------+--------------------+------------+-----------------+
|StockCode|         Description|TotalRevenue|TotalQuantitySold|
+---------+--------------------+------------+-----------------+
|   85123A|WHITE HANGING HEA...|   151624.31|          56915.0|
|    22423|REGENCY CAKESTAND...|   143893.35|          12497.0|
|        M|              Manual|    97852.82|            891.0|
|    84879|ASSORTED COLOUR B...|    70493.83|          44551.0|
|   85099B|JUMBO BAG RED RET...|     51759.3|          29578.0|
|     POST|             POSTAGE|    48741.08|           2212.0|
|    84347|ROTATING SILVER A...|    40186.65|          21591.0|
|    22086|PAPER CHAIN KIT 5...|     36933.5|          13860.0|
|    47566|       PARTY BUNTING|     35035.9|           8316.0|
|   15056N|EDWARDIAN PARASOL...|    34044.75|           7201.0|
+---------+--------------------+------------+-----------------+
only showing top 10 rows



In [6]:
max_date = df_clean.agg(F.max("InvoiceDate")).collect()[0][0]

gold_customer_rfm = df_clean \
    .groupBy("Customer_ID") \
    .agg(
        F.datediff(F.lit(max_date), 
            F.max("InvoiceDate")).alias("Recency"),
        F.countDistinct("Invoice").alias("Frequency"),
        F.round(F.sum("TotalAmount"), 2).alias("Monetary")
    ).orderBy("Recency")

gold_customer_rfm.show(10)

+-----------+-------+---------+--------+
|Customer_ID|Recency|Frequency|Monetary|
+-----------+-------+---------+--------+
|      12847|   null|        1|  664.39|
|      13610|   null|        7|  986.47|
|      13192|   null|        4|  1403.6|
|      15555|   null|       22| 6293.19|
|      14157|   null|        3|   518.2|
|      15271|   null|       10| 1915.44|
|      16250|   null|        2|   369.9|
|      16576|   null|        2|  474.27|
|      15574|   null|        3|  666.14|
|      13282|   null|        1|  127.05|
+-----------+-------+---------+--------+
only showing top 10 rows



In [7]:
import os

# Create gold directories if they don't exist
os.makedirs(GOLD_PATH + "/monthly_revenue", exist_ok=True)
os.makedirs(GOLD_PATH + "/top_products", exist_ok=True)
os.makedirs(GOLD_PATH + "/customer_rfm", exist_ok=True)

# Use forward slashes for Spark paths
gold_path = GOLD_PATH.replace("\\", "/")

gold_monthly_revenue.write.format("delta") \
    .mode("overwrite") \
    .save(f"{gold_path}/monthly_revenue")

gold_top_products.write.format("delta") \
    .mode("overwrite") \
    .save(f"{gold_path}/top_products")

gold_customer_rfm.write.format("delta") \
    .mode("overwrite") \
    .save(f"{gold_path}/customer_rfm")

print("Gold layer written successfully ✅")
print(f"Check your output at: {GOLD_PATH}")

Gold layer written successfully ✅
Check your output at: D:\DS\visualization\retail_pipeline\data\gold


In [ ]:
# Test writing a simple parquet file first (no Delta)
test_df = spark.createDataFrame([(1, "test")], ["id", "value"])

test_path = r"D:/DS/visualization/retail_pipeline/data/gold/test"

test_df.write \
    .mode("overwrite") \
    .parquet(test_path)

print("Test write successful ✅")